# Shrinkage LDA (192 features)
Loads `checkpoints/classification_data.npz` → saves `results/result_shrinkage_lda.npz`

In [1]:
import numpy as np, os, time, copy
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, classification_report

import importlib.util
spec = importlib.util.spec_from_file_location("cfg", "00_config.py")
cfg = importlib.util.module_from_spec(spec); spec.loader.exec_module(cfg)
for a in dir(cfg):
    if not a.startswith('_'): globals()[a] = getattr(cfg, a)

data = np.load(CKPT['clf_data'])
X = data['X_rich']
y = data['y']; groups = data['groups']
subjects = np.load(CKPT['subjects'], allow_pickle=True).tolist()
print(f"X={X.shape} tgt={sum(y==1)} ntgt={sum(y==0)}")

X=(3600, 192) tgt=600 ntgt=3000


In [2]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
])

In [3]:
# LOSO
t0 = time.time()
logo = LeaveOneGroupOut()
yt_all, yp_all, scores = [], [], []

for fold, (tr, te) in enumerate(logo.split(X, y, groups)):
    p = copy.deepcopy(pipe)
    p.fit(X[tr], y[tr])
    yp = p.predict(X[te])
    yt_all.extend(y[te]); yp_all.extend(yp)
    ba = balanced_accuracy_score(y[te], yp)
    scores.append(ba)
    print(f"  {subjects[fold]}: {ba:.3f}")

scores = np.array(scores)
print(f"\nmean={np.mean(scores):.3f}±{np.std(scores):.3f} ({time.time()-t0:.0f}s)")

  sub-010: 0.917
  sub-011: 0.608
  sub-012: 0.605
  sub-013: 0.805
  sub-014: 0.813
  sub-015: 0.862
  sub-016: 0.785
  sub-019: 0.688
  sub-020: 0.723
  sub-021: 0.732

mean=0.754±0.097 (1s)


In [4]:
np.savez(result_path('shrinkage_lda'),
    method='shrinkage_lda', per_subject_scores=scores,
    y_true=np.array(yt_all), y_pred=np.array(yp_all),
    elapsed=time.time()-t0, subjects=subjects)
print(f"saved → {result_path('shrinkage_lda')}")
print(classification_report(np.array(yt_all), np.array(yp_all), target_names=['Non-tgt','Target']))

saved → .\results\result_shrinkage_lda.npz
              precision    recall  f1-score   support

     Non-tgt       0.91      0.96      0.93      3000
      Target       0.71      0.55      0.62       600

    accuracy                           0.89      3600
   macro avg       0.81      0.75      0.78      3600
weighted avg       0.88      0.89      0.88      3600

